# Sistema de Portafolio con Consciencia Situacional
### Basado en Dai et al. (2010) — Optimal Trend Following Trading Rules

**Arquitectura:**
- Capa 0: Régimen Macro (este notebook)
- Capa 1: Matriz de Correlaciones Rolling
- Capa 2: Filtro de Wonham (detección de régimen bull/bear)
- Capa 3: Filtro de Noticias (Claude API)
- Capa 4: Optimización + Prueba de Estrés

**Datos:** yfinance + FRED (gratuitos)  
**Frecuencia de rebalanceo:** Mensual  
**Horizonte de análisis:** 6 meses

---
## CELDA 1 — Instalación de dependencias
Ejecutar solo la primera vez

In [ ]:
# Instalación de librerías necesarias
# Ejecutar solo una vez; comentar después
%pip install yfinance pandas numpy matplotlib seaborn scipy pandas-datareader fredapi plotly ipywidgets --quiet

---
## CELDA 2 — Imports y configuración global

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta
import pandas_datareader.data as web

# Estilo visual
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)

# ─────────────────────────────────────────────
# CONFIGURACIÓN GLOBAL — ajustar según cliente
# ─────────────────────────────────────────────
FECHA_FIN   = datetime.today().strftime('%Y-%m-%d')
FECHA_INICIO_LARGO  = '2018-01-01'   # 6+ años para calibrar Wonham
FECHA_INICIO_CORTO  = '2024-01-01'   # ~18 meses para correlaciones
VENTANA_CORR        = 60             # días para correlación rolling
VENTANA_WONHAM      = 252 * 3        # días para calibrar parámetros bull/bear

# Universo de activos base (Capa 0 y Capa 1)
ACTIVOS_MACRO = {
    'SP500'  : '^GSPC',
    'NDX100' : '^NDX',
    'DXY'    : 'DX-Y.NYB',
    'ORO'    : 'GC=F',
    'BTC'    : 'BTC-USD',
    'TNX'    : '^TNX',    # Yield 10 años USA
    'VIX'    : '^VIX',
    'PETROLEO': 'CL=F',
}

print(f'✅ Configuración cargada | Fecha de análisis: {FECHA_FIN}')
print(f'   Ventana correlaciones: {VENTANA_CORR} días')
print(f'   Ventana calibración Wonham: {VENTANA_WONHAM} días (~{VENTANA_WONHAM//252} años)')

---
## CELDA 3 — Descarga de precios de activos macro

In [ ]:
def descargar_activos(tickers_dict, inicio, fin):
    """
    Descarga precios de cierre ajustados para un diccionario de activos.
    Retorna DataFrame con nombres legibles como columnas.
    """
    precios = {}
    errores = []
    
    for nombre, ticker in tickers_dict.items():
        try:
            datos = yf.download(ticker, start=inicio, end=fin,
                                auto_adjust=True, progress=False)
            if len(datos) > 10:
                precios[nombre] = datos['Close'].squeeze()
                print(f'  ✅ {nombre:12} ({ticker:12}) | {len(datos)} registros')
            else:
                errores.append(nombre)
                print(f'  ⚠️  {nombre:12} ({ticker:12}) | datos insuficientes')
        except Exception as e:
            errores.append(nombre)
            print(f'  ❌ {nombre:12} ({ticker:12}) | Error: {e}')
    
    df = pd.DataFrame(precios)
    df.index = pd.to_datetime(df.index)
    df = df.dropna(how='all')
    
    if errores:
        print(f'\n⚠️  Activos con error: {errores}')
    
    return df


print('📥 Descargando activos macro (largo plazo)...')
precios_largo = descargar_activos(ACTIVOS_MACRO, FECHA_INICIO_LARGO, FECHA_FIN)

print(f'\n📥 Descargando activos macro (corto plazo)...')
precios_corto = descargar_activos(ACTIVOS_MACRO, FECHA_INICIO_CORTO, FECHA_FIN)

print(f'\n📊 Dataset largo: {precios_largo.shape[0]} días × {precios_largo.shape[1]} activos')
print(f'📊 Dataset corto: {precios_corto.shape[0]} días × {precios_corto.shape[1]} activos')
precios_corto.tail(3)

---
## CELDA 4 — Descarga de indicadores macro desde FRED
FRED es la base de datos del Federal Reserve Bank de St. Louis — acceso 100% gratuito sin API key para la mayoría de series.

In [ ]:
# Series FRED (acceso gratuito sin API key via pandas-datareader)
SERIES_FRED = {
    'FED_FUNDS'    : 'FEDFUNDS',      # Tasa Fed Funds (mensual)
    'CPI_YOY'      : 'CPIAUCSL',      # CPI (mensual, ajustar a YoY)
    'PPI_YOY'      : 'PPIACO',        # PPI bienes (mensual)
    'DESEMPLEO'    : 'UNRATE',        # Tasa desempleo (mensual)
    'M2'           : 'M2SL',          # Masa monetaria M2 (mensual)
    'SPREAD_10_2'  : 'T10Y2Y',        # Spread 10yr - 2yr (diario)
    'CRED_SPREAD'  : 'BAMLH0A0HYM2',  # High Yield spread (diario)
    'BREAKEVEN_5'  : 'T5YIE',         # Inflación implícita 5 años
}

macro_fred = {}
errores_fred = []

print('📥 Descargando indicadores macro de FRED...')
for nombre, codigo in SERIES_FRED.items():
    try:
        serie = web.DataReader(codigo, 'fred',
                               start=FECHA_INICIO_LARGO,
                               end=FECHA_FIN)
        macro_fred[nombre] = serie.squeeze()
        print(f'  ✅ {nombre:15} ({codigo:15}) | {len(serie)} registros')
    except Exception as e:
        errores_fred.append(nombre)
        print(f'  ❌ {nombre:15} ({codigo:15}) | {str(e)[:50]}')

# Calcular variaciones YoY para CPI y PPI
if 'CPI_YOY' in macro_fred:
    macro_fred['CPI_YOY'] = macro_fred['CPI_YOY'].pct_change(12) * 100
if 'PPI_YOY' in macro_fred:
    macro_fred['PPI_YOY'] = macro_fred['PPI_YOY'].pct_change(12) * 100

df_fred = pd.DataFrame(macro_fred)
print(f'\n📊 FRED Dataset: {df_fred.shape[0]} filas × {df_fred.shape[1]} series')
print('\nÚltimas lecturas disponibles:')
print(df_fred.dropna().tail(3).to_string())

---
## CELDA 5 — Capa 0: Scorecard de Régimen Macro

Basado en los cuatro cuadrantes de Dalio (All Weather), clasificamos el régimen actual usando
indicadores de **crecimiento** e **inflación**. Este scorecard es la aproximación discreta del
filtro de Wonham — la probabilidad `p_t` de estar en régimen alcista.

In [ ]:
def calcular_scorecard_regimen(df_fred, precios_largo):
    """
    Calcula el scorecard de régimen macro en tres dimensiones:
    1. Crecimiento (0-100)
    2. Inflación (0-100, donde 100 = inflación alta)
    3. Liquidez (0-100, donde 100 = condiciones laxas)
    
    Retorna dict con scores y clasificación de régimen.
    """
    scores = {}
    senales = {}
    
    # ── DIMENSIÓN 1: CRECIMIENTO ──────────────────────────────────────────
    
    # SP500 momentum (precio actual vs MA de 200 días)
    sp500 = precios_largo['SP500'].dropna()
    ma200 = sp500.rolling(200).mean()
    sp_sobre_ma200 = sp500.iloc[-1] > ma200.iloc[-1]
    senales['SP500_sobre_MA200'] = sp_sobre_ma200
    
    # Rendimiento SP500 últimos 12 meses
    sp_ret_12m = (sp500.iloc[-1] / sp500.iloc[-253] - 1) * 100 if len(sp500) > 253 else 0
    senales['SP500_ret_12m'] = round(sp_ret_12m, 2)
    
    # Spread 10yr-2yr (curva de tasas)
    spread = None
    if 'SPREAD_10_2' in df_fred.columns:
        spread_serie = df_fred['SPREAD_10_2'].dropna()
        if len(spread_serie) > 0:
            spread = spread_serie.iloc[-1]
            senales['Curva_10_2'] = round(spread, 3)
            senales['Curva_positiva'] = spread > 0
    
    # Desempleo (bajo = bueno)
    desempleo = None
    if 'DESEMPLEO' in df_fred.columns:
        desempleo_serie = df_fred['DESEMPLEO'].dropna()
        if len(desempleo_serie) > 0:
            desempleo = desempleo_serie.iloc[-1]
            desempleo_12m_atras = desempleo_serie.iloc[-13] if len(desempleo_serie) > 13 else desempleo
            senales['Desempleo_actual'] = round(desempleo, 1)
            senales['Desempleo_subiendo'] = desempleo > desempleo_12m_atras
    
    # Score de crecimiento: suma de señales positivas
    score_crecimiento = 0
    if sp_sobre_ma200:                            score_crecimiento += 30
    if sp_ret_12m > 0:                            score_crecimiento += 25
    if spread is not None and spread > 0:         score_crecimiento += 25
    if desempleo is not None and desempleo < 5.0: score_crecimiento += 20
    scores['Crecimiento'] = score_crecimiento
    
    # ── DIMENSIÓN 2: INFLACIÓN ────────────────────────────────────────────
    
    # CPI YoY
    cpi = None
    if 'CPI_YOY' in df_fred.columns:
        cpi_serie = df_fred['CPI_YOY'].dropna()
        if len(cpi_serie) > 0:
            cpi = cpi_serie.iloc[-1]
            cpi_3m_atras = cpi_serie.iloc[-4] if len(cpi_serie) > 4 else cpi
            senales['CPI_YoY'] = round(cpi, 2)
            senales['CPI_acelerando'] = cpi > cpi_3m_atras
    
    # PPI YoY
    ppi = None
    if 'PPI_YOY' in df_fred.columns:
        ppi_serie = df_fred['PPI_YOY'].dropna()
        if len(ppi_serie) > 0:
            ppi = ppi_serie.iloc[-1]
            senales['PPI_YoY'] = round(ppi, 2)
    
    # Breakeven de inflación
    breakeven = None
    if 'BREAKEVEN_5' in df_fred.columns:
        be_serie = df_fred['BREAKEVEN_5'].dropna()
        if len(be_serie) > 0:
            breakeven = be_serie.iloc[-1]
            senales['Breakeven_5yr'] = round(breakeven, 2)
    
    # DXY (dólar fuerte = desinflacionario)
    dxy = precios_largo['DXY'].dropna()
    dxy_ret_3m = (dxy.iloc[-1] / dxy.iloc[-65] - 1) * 100 if len(dxy) > 65 else 0
    senales['DXY_ret_3m'] = round(dxy_ret_3m, 2)
    
    # Score de inflación (alto = inflación alta)
    score_inflacion = 0
    if cpi is not None and cpi > 3.0:     score_inflacion += 30
    if cpi is not None and cpi > 2.0:     score_inflacion += 20
    if ppi is not None and ppi > 3.0:     score_inflacion += 20
    if breakeven is not None and breakeven > 2.5: score_inflacion += 20
    if dxy_ret_3m < -2:                   score_inflacion += 10  # dólar débil = más inflación
    scores['Inflacion'] = min(score_inflacion, 100)
    
    # ── DIMENSIÓN 3: LIQUIDEZ ─────────────────────────────────────────────
    
    # Fed Funds Rate
    fed = None
    if 'FED_FUNDS' in df_fred.columns:
        fed_serie = df_fred['FED_FUNDS'].dropna()
        if len(fed_serie) > 0:
            fed = fed_serie.iloc[-1]
            fed_12m_atras = fed_serie.iloc[-13] if len(fed_serie) > 13 else fed
            senales['Fed_Funds'] = round(fed, 2)
            senales['Fed_bajando'] = fed < fed_12m_atras
    
    # Credit spread (bajo = liquidez abundante)
    cred = None
    if 'CRED_SPREAD' in df_fred.columns:
        cred_serie = df_fred['CRED_SPREAD'].dropna()
        if len(cred_serie) > 0:
            cred = cred_serie.iloc[-1]
            senales['HY_Spread'] = round(cred, 2)
    
    # VIX (bajo = liquidez / risk-on)
    vix = precios_largo['VIX'].dropna()
    vix_actual = vix.iloc[-1]
    senales['VIX'] = round(vix_actual, 1)
    
    # M2 growth
    m2_growth = None
    if 'M2' in df_fred.columns:
        m2_serie = df_fred['M2'].dropna()
        if len(m2_serie) > 13:
            m2_growth = (m2_serie.iloc[-1] / m2_serie.iloc[-13] - 1) * 100
            senales['M2_growth_YoY'] = round(m2_growth, 2)
    
    # Score de liquidez (alto = condiciones favorables)
    score_liquidez = 0
    if fed is not None and fed < 3.0:     score_liquidez += 25
    elif fed is not None and fed < 5.0:   score_liquidez += 10
    if cred is not None and cred < 4.0:   score_liquidez += 25
    elif cred is not None and cred < 6.0: score_liquidez += 10
    if vix_actual < 20:                   score_liquidez += 25
    elif vix_actual < 30:                 score_liquidez += 10
    if m2_growth is not None and m2_growth > 2: score_liquidez += 25
    scores['Liquidez'] = score_liquidez
    
    # ── CLASIFICACIÓN DE RÉGIMEN ──────────────────────────────────────────
    crec_alto = scores['Crecimiento'] >= 50
    infl_alta = scores['Inflacion'] >= 50
    
    if crec_alto and not infl_alta:
        regimen = 'GOLDILOCKS'
        descripcion = 'Crecimiento con inflación controlada — favorable para renta variable growth'
        activos_favorecidos = ['NDX100', 'SP500', 'acciones_growth', 'crypto']
    elif crec_alto and infl_alta:
        regimen = 'REFLACION'
        descripcion = 'Crecimiento con inflación alta — favorable para commodities, value, defensa'
        activos_favorecidos = ['ORO', 'PETROLEO', 'acciones_value', 'defensa', 'TIPS']
    elif not crec_alto and infl_alta:
        regimen = 'ESTANFLACION'
        descripcion = 'Estancamiento con inflación — desfavorable para casi todo excepto oro y cash'
        activos_favorecidos = ['ORO', 'cash', 'commodities_defensivos']
    else:
        regimen = 'DEFLACION_RECESION'
        descripcion = 'Desaceleración con inflación baja — favorable para bonos largos y oro'
        activos_favorecidos = ['bonos_largos', 'ORO', 'defensivos']
    
    # p_t estimada (proxy del filtro de Wonham)
    p_t = (scores['Crecimiento'] * 0.5 + scores['Liquidez'] * 0.5) / 100
    
    return {
        'scores': scores,
        'senales': senales,
        'regimen': regimen,
        'descripcion': descripcion,
        'activos_favorecidos': activos_favorecidos,
        'p_t_estimada': round(p_t, 4)
    }


resultado_macro = calcular_scorecard_regimen(df_fred, precios_largo)

print('=' * 65)
print('  SCORECARD DE RÉGIMEN MACRO — CAPA 0')
print('=' * 65)
print(f"\n  📊 Score Crecimiento : {resultado_macro['scores']['Crecimiento']:>3}/100")
print(f"  📈 Score Inflación   : {resultado_macro['scores']['Inflacion']:>3}/100")
print(f"  💧 Score Liquidez    : {resultado_macro['scores']['Liquidez']:>3}/100")
print(f"\n  🎯 RÉGIMEN ACTUAL    : {resultado_macro['regimen']}")
print(f"  📝 {resultado_macro['descripcion']}")
print(f"\n  ✅ Activos favorecidos: {', '.join(resultado_macro['activos_favorecidos'])}")
print(f"\n  🔢 p_t estimada (proxy Wonham): {resultado_macro['p_t_estimada']}")
print(f"     → Interpretación: probabilidad de mercado alcista = {resultado_macro['p_t_estimada']*100:.1f}%")

print('\n' + '-' * 65)
print('  SEÑALES INDIVIDUALES')
print('-' * 65)
for k, v in resultado_macro['senales'].items():
    simbolo = '✅' if v is True else ('❌' if v is False else '📌')
    print(f'  {simbolo}  {k:25}: {v}')

---
## CELDA 6 — Visualización del Scorecard de Régimen

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

colores_regimen = {
    'GOLDILOCKS': '#27ae60',
    'REFLACION': '#e67e22',
    'ESTANFLACION': '#e74c3c',
    'DEFLACION_RECESION': '#2980b9'
}
color_actual = colores_regimen.get(resultado_macro['regimen'], '#7f8c8d')

# ── Cuadrante Dalio ──────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
cuadrantes = {
    'GOLDILOCKS\n(Crec↑ Infl↓)'   : (1, 1),
    'REFLACION\n(Crec↑ Infl↑)'    : (0, 1),
    'ESTANFLACION\n(Crec↓ Infl↑)' : (0, 0),
    'DEFLACION\n(Crec↓ Infl↓)'    : (1, 0)
}
colores_cuad = ['#27ae60', '#e67e22', '#e74c3c', '#2980b9']
posiciones   = [(0.5, 0.75), (-0.5, 0.75), (-0.5, 0.25), (0.5, 0.25)]

for (nombre, pos), color, (px, py) in zip(cuadrantes.items(),
                                            colores_cuad, posiciones):
    alpha = 0.85 if nombre.split('\n')[0] in resultado_macro['regimen'] else 0.2
    borde = 3 if nombre.split('\n')[0] in resultado_macro['regimen'] else 1
    rect = plt.Rectangle((px - 0.5, py - 0.25), 1, 0.5,
                          facecolor=color, alpha=alpha,
                          edgecolor='black', linewidth=borde)
    ax1.add_patch(rect)
    ax1.text(px, py, nombre, ha='center', va='center',
             fontsize=7.5, fontweight='bold', color='white')

ax1.set_xlim(-1, 1)
ax1.set_ylim(0, 1)
ax1.axvline(0, color='black', linewidth=1.5)
ax1.axhline(0.5, color='black', linewidth=1.5)
ax1.set_xlabel('Inflación →', fontsize=9)
ax1.set_ylabel('← Deflación    Crecimiento →', fontsize=9)
ax1.set_title('Cuadrante Macro\n(All Weather — Dalio)', fontsize=10, fontweight='bold')
ax1.set_xticks([])
ax1.set_yticks([])

# ── Gauge scores ─────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
dimensiones = list(resultado_macro['scores'].keys())
valores     = list(resultado_macro['scores'].values())
colores_bar = ['#27ae60' if v >= 50 else '#e74c3c' for v in valores]

bars = ax2.barh(dimensiones, valores, color=colores_bar, edgecolor='white', height=0.5)
ax2.set_xlim(0, 100)
ax2.axvline(50, color='black', linewidth=1.5, linestyle='--', alpha=0.5)
ax2.set_title('Scores por Dimensión', fontsize=10, fontweight='bold')
ax2.set_xlabel('Score (0-100)')
for bar, val in zip(bars, valores):
    ax2.text(val + 2, bar.get_y() + bar.get_height()/2,
             f'{val}', va='center', fontweight='bold')

# ── SP500 + MA200 ─────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
sp = precios_corto['SP500'].dropna()
ma200_corto = precios_largo['SP500'].rolling(200).mean()
ma200_recort = ma200_corto[ma200_corto.index >= FECHA_INICIO_CORTO]

ax3.plot(sp.index, sp.values, color='#2c3e50', linewidth=1.5, label='SP500')
ax3.plot(ma200_recort.index, ma200_recort.values,
         color='#e74c3c', linewidth=1.5, linestyle='--', label='MA200')
ax3.fill_between(sp.index, sp.values, ma200_recort.reindex(sp.index),
                 where=sp.values > ma200_recort.reindex(sp.index).values,
                 alpha=0.15, color='green', label='Sobre MA200')
ax3.set_title('SP500 vs MA200', fontsize=10, fontweight='bold')
ax3.legend(fontsize=8)
ax3.tick_params(axis='x', rotation=30)

# ── CPI e Inflación ───────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
if 'CPI_YOY' in df_fred.columns and 'PPI_YOY' in df_fred.columns:
    cpi_p = df_fred['CPI_YOY'].dropna()
    ppi_p = df_fred['PPI_YOY'].dropna()
    cpi_rec = cpi_p[cpi_p.index >= FECHA_INICIO_CORTO]
    ppi_rec = ppi_p[ppi_p.index >= FECHA_INICIO_CORTO]
    ax4.plot(cpi_rec.index, cpi_rec.values, color='#e74c3c', linewidth=2, label='CPI YoY%')
    ax4.plot(ppi_rec.index, ppi_rec.values, color='#e67e22', linewidth=2,
             linestyle='--', label='PPI YoY%')
    ax4.axhline(2.0, color='gray', linewidth=1, linestyle=':', label='Meta 2%')
    ax4.axhline(0, color='black', linewidth=0.8)
    ax4.set_title('Inflación CPI y PPI', fontsize=10, fontweight='bold')
    ax4.legend(fontsize=8)
    ax4.set_ylabel('%')
    ax4.tick_params(axis='x', rotation=30)

# ── Curva de tasas (Spread 10-2) ──────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
if 'SPREAD_10_2' in df_fred.columns:
    spread_p = df_fred['SPREAD_10_2'].dropna()
    spread_rec = spread_p[spread_p.index >= FECHA_INICIO_CORTO]
    ax5.plot(spread_rec.index, spread_rec.values, color='#8e44ad', linewidth=2)
    ax5.fill_between(spread_rec.index, spread_rec.values, 0,
                     where=spread_rec.values > 0,
                     alpha=0.2, color='green', label='Normal (positivo)')
    ax5.fill_between(spread_rec.index, spread_rec.values, 0,
                     where=spread_rec.values < 0,
                     alpha=0.3, color='red', label='Invertida (negativo)')
    ax5.axhline(0, color='black', linewidth=1.5)
    ax5.set_title('Curva de Tasas\n(Spread 10yr - 2yr)', fontsize=10, fontweight='bold')
    ax5.legend(fontsize=8)
    ax5.set_ylabel('% puntos')
    ax5.tick_params(axis='x', rotation=30)

# ── VIX ───────────────────────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
vix_corto = precios_corto['VIX'].dropna()
ax6.plot(vix_corto.index, vix_corto.values, color='#e74c3c', linewidth=1.5)
ax6.axhline(20, color='orange', linewidth=1.5, linestyle='--', label='Umbral 20 (cautela)')
ax6.axhline(30, color='red', linewidth=1.5, linestyle='--', label='Umbral 30 (miedo)')
ax6.fill_between(vix_corto.index, vix_corto.values,
                 alpha=0.2, color='#e74c3c')
ax6.set_title('VIX — Índice de Volatilidad', fontsize=10, fontweight='bold')
ax6.legend(fontsize=8)
ax6.set_ylabel('VIX')
ax6.tick_params(axis='x', rotation=30)

# Título general
fig.suptitle(
    f'CAPA 0 — Régimen Macro: {resultado_macro["regimen"]}  '
    f'| p_t = {resultado_macro["p_t_estimada"]}  '
    f'| {datetime.today().strftime("%d/%m/%Y")}',
    fontsize=13, fontweight='bold', color=color_actual, y=1.01
)

plt.savefig('capa0_regimen_macro.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como capa0_regimen_macro.png')

---
## CELDA 7 — Capa 1: Retornos y Matriz de Correlaciones

In [ ]:
# ── Calcular retornos logarítmicos diarios ────────────────────────────────
retornos = np.log(precios_corto / precios_corto.shift(1)).dropna()

print('📊 Estadísticas de retornos diarios:')
stats_retornos = pd.DataFrame({
    'Retorno_anual_%'  : (retornos.mean() * 252 * 100).round(2),
    'Volat_anual_%'    : (retornos.std() * np.sqrt(252) * 100).round(2),
    'Sharpe_aprox'     : ((retornos.mean() * 252) / (retornos.std() * np.sqrt(252))).round(3),
    'Sesgo'            : retornos.skew().round(3),
    'Kurtosis'         : retornos.kurtosis().round(3),
    'Max_caida_%'      : (retornos.min() * 100).round(2),
}).T

print(stats_retornos.to_string())
print(f'\n📅 Período: {retornos.index[0].strftime("%d/%m/%Y")} — {retornos.index[-1].strftime("%d/%m/%Y")}')
print(f'📆 Total de días: {len(retornos)}')

---
## CELDA 8 — Matriz de correlaciones estática y rolling

In [ ]:
# ── Correlación estática (período completo) ───────────────────────────────
corr_estatica = retornos.corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap correlación estática
mask = np.triu(np.ones_like(corr_estatica, dtype=bool), k=1)
sns.heatmap(
    corr_estatica,
    ax=axes[0],
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
axes[0].set_title(
    f'Correlación Estática\n({FECHA_INICIO_CORTO} — {FECHA_FIN})',
    fontsize=11, fontweight='bold'
)
axes[0].tick_params(axis='x', rotation=45)

# ── Correlación rolling SP500 vs resto ───────────────────────────────────
activos_vs_sp = [c for c in retornos.columns if c != 'SP500']
colores_rolling = plt.cm.tab10(np.linspace(0, 1, len(activos_vs_sp)))

for activo, color in zip(activos_vs_sp, colores_rolling):
    corr_roll = retornos['SP500'].rolling(VENTANA_CORR).corr(retornos[activo])
    axes[1].plot(corr_roll.index, corr_roll.values,
                 label=activo, color=color, linewidth=1.5, alpha=0.85)

axes[1].axhline(0, color='black', linewidth=1.5, linestyle='--')
axes[1].axhline(0.7, color='red', linewidth=1, linestyle=':', alpha=0.6,
                label='Umbral concentración (0.7)')
axes[1].axhline(-0.5, color='green', linewidth=1, linestyle=':', alpha=0.6,
                label='Umbral cobertura (-0.5)')
axes[1].set_title(
    f'Correlación Rolling vs SP500\n(ventana {VENTANA_CORR} días)',
    fontsize=11, fontweight='bold'
)
axes[1].set_ylabel('Correlación de Pearson')
axes[1].legend(fontsize=8, loc='upper left')
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylim(-1.1, 1.1)

plt.suptitle('CAPA 1 — Mapa de Correlaciones de Activos Macro',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('capa1_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Guardado como capa1_correlaciones.png')

---
## CELDA 9 — Correlaciones actuales y alertas de concentración

In [ ]:
# Correlaciones rolling al día de hoy (últimos VENTANA_CORR días)
retornos_recientes = retornos.tail(VENTANA_CORR)
corr_actual = retornos_recientes.corr()

print(f'📊 CORRELACIONES ACTUALES (últimos {VENTANA_CORR} días trading)')
print('=' * 55)
print(corr_actual.round(3).to_string())

# ── Alertas automáticas ───────────────────────────────────────────────────
print('\n⚠️  ALERTAS DE CORRELACIÓN')
print('-' * 55)

alertas_concentracion = []
alertas_cobertura     = []

pares_revisados = set()
for i in corr_actual.columns:
    for j in corr_actual.columns:
        if i >= j:
            continue
        par = (i, j)
        if par in pares_revisados:
            continue
        pares_revisados.add(par)
        
        c = corr_actual.loc[i, j]
        if c >= 0.85:
            alertas_concentracion.append((i, j, c))
            print(f'  🔴 CONCENTRACIÓN ALTA: {i} ↔ {j} = {c:.3f}')
        elif c <= -0.6:
            alertas_cobertura.append((i, j, c))
            print(f'  🟢 COBERTURA NATURAL:  {i} ↔ {j} = {c:.3f}')

if not alertas_concentracion and not alertas_cobertura:
    print('  ✅ Sin alertas extremas en el período actual')

# ── Activos con mejor diversificación respecto al SP500 ───────────────────
if 'SP500' in corr_actual.columns:
    corr_vs_sp500 = corr_actual['SP500'].drop('SP500').sort_values()
    print(f'\n📌 Correlación vs SP500 (menor = mejor diversificador):')
    for activo, corr_val in corr_vs_sp500.items():
        barra = '█' * int(abs(corr_val) * 20)
        signo = '+' if corr_val >= 0 else '-'
        print(f'  {activo:12} {signo}{barra:<20} {corr_val:.3f}')

---
## CELDA 10 — Resumen ejecutivo de Capas 0 y 1
Output estructurado para alimentar la Capa 2 (Wonham) y Capa 3 (selección)

In [ ]:
# Estado persistente del análisis — se usará en las siguientes capas
ESTADO_SISTEMA = {
    # Capa 0
    'fecha_analisis'       : FECHA_FIN,
    'regimen_macro'        : resultado_macro['regimen'],
    'score_crecimiento'    : resultado_macro['scores']['Crecimiento'],
    'score_inflacion'      : resultado_macro['scores']['Inflacion'],
    'score_liquidez'       : resultado_macro['scores']['Liquidez'],
    'p_t_estimada'         : resultado_macro['p_t_estimada'],
    'activos_favorecidos'  : resultado_macro['activos_favorecidos'],
    'senales_macro'        : resultado_macro['senales'],
    
    # Capa 1
    'corr_actual'          : corr_actual,
    'alertas_concentracion': alertas_concentracion,
    'alertas_cobertura'    : alertas_cobertura,
    'stats_retornos'       : stats_retornos,
    
    # Datos para Wonham (Capa 2)
    'precios_largo'        : precios_largo,
    'retornos'             : retornos,
}

print('╔══════════════════════════════════════════════════════════════╗')
print('║         RESUMEN EJECUTIVO — CAPAS 0 Y 1 COMPLETADAS         ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Fecha:          {FECHA_FIN:<43}║')
print(f'║  Régimen:        {resultado_macro["regimen"]:<43}║')
print(f'║  p_t estimada:   {resultado_macro["p_t_estimada"]:<43}║')
print(f'║  Score Crec:     {resultado_macro["scores"]["Crecimiento"]}/100{" "*39}║')
print(f'║  Score Infl:     {resultado_macro["scores"]["Inflacion"]}/100{" "*39}║')
print(f'║  Score Liq:      {resultado_macro["scores"]["Liquidez"]}/100{" "*39}║')
print(f'║  Alertas conc:   {len(alertas_concentracion):<43}║')
print(f'║  Alertas cob:    {len(alertas_cobertura):<43}║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  SIGUIENTE: Capa 2 — Filtro de Wonham (calibración HMM)     ║')
print('╚══════════════════════════════════════════════════════════════╝')

print('\n✅ Variable ESTADO_SISTEMA guardada — disponible para Capa 2')